## Chatbot
 Design and Implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.


import 


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv() ## loading all the environment variables from .env file

True

In [3]:
groq_api_key = os.getenv("GROQ_API_KEY")

In [9]:

from langchain_groq import ChatGroq

# the above line is used to import the ChatGroq class from the langchain_groq library, which allows us to interact with the Groq API for chat-based applications.

model = ChatGroq(
    model = "openai/gpt-oss-120b",
    groq_api_key = groq_api_key
)

model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000019BB9CD0D60>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000019BB9CD12A0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [13]:
from langchain_core.messages import HumanMessage

response1 = model.invoke(
    [
        HumanMessage(
            content="Hey, I am Pratham Raj and I am building a chatbot that can recall the previous messages in the conversation. Can you help me with that?"
        )
    ]
)

In [14]:
print(response1.content)

### Hey Pratham! 👋  
Building a chatbot that can **remember** what was said earlier in the same conversation is a common requirement, and there are a few patterns you can follow depending on the complexity you need and the tools you’re comfortable with.

Below is a step‑by‑step guide that covers:

1. **Why “memory” matters**  
2. **Simple in‑memory rolling window** (quick‑start)  
3. **Long‑term conversation history** (persistent storage)  
4. **Semantic / vector‑based recall** (search‑by‑meaning)  
5. **Putting it all together with a minimal example** (Python + OpenAI API)  
6. **Tips & best‑practices**  

Feel free to cherry‑pick the parts that fit your project!

---

## 1. The Problem – What “recall” Means

| Goal | Typical implementation |
|------|------------------------|
| **Echo recent turns** (e.g., last 5 messages) | Keep a list/queue in RAM and prepend it to every request. |
| **Persist conversation across sessions** (user logs back in later) | Store the full transcript in a 

In [15]:
from langchain_core.messages import HumanMessage

response2 = model.invoke(
    [
        HumanMessage(
            content="Hey, I am Pratham Raj and Currently I am software engineer at Mindsprint"
        )
    ]
)

In [16]:
response2


AIMessage(content='Hey Pratham! 👋 Great to meet you. How can I assist you today?', additional_kwargs={'reasoning_content': 'The user says: "Hey, I am Pratham Raj and Currently I am software engineer at Mindsprint". Likely they want a response. Could be a greeting, maybe ask how can help. Should respond politely. No disallowed content. So respond friendly.'}, response_metadata={'token_usage': {'completion_tokens': 81, 'prompt_tokens': 87, 'total_tokens': 168, 'completion_time': 0.172938863, 'completion_tokens_details': {'reasoning_tokens': 54}, 'prompt_time': 0.003726123, 'prompt_tokens_details': None, 'queue_time': 0.046037836, 'total_time': 0.176664986}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_a09bde29de', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d3a8d-763a-7662-9139-ab4d3042a265-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 87, 'output_tokens': 81, 'total_tokens

In [17]:
print(response2.content)

Hey Pratham! 👋 Great to meet you. How can I assist you today?


In [18]:
from langchain_core.messages import AIMessage
response3 = model.invoke(
    [
        HumanMessage(
            content="Hey, I am Pratham Raj and Currently I am software engineer at Mindsprint"
        ),
        AIMessage(
            content="Hey Pratham! 👋 Great to meet you. How can I assist you today?"
        ),
        HumanMessage(
            content="Who am I and what is my job?"
        )
    ]
)

In [19]:
response3.content

'You’re **Pratham\u202fRaj**, and you work as a **software engineer** at **Mindsprint**.'

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [21]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

## when multiple user using the same chatbot , how we can differentiate that one user chat session different from another user chat session.
## session id is used to differentiate between different user chat sessions. We can use session id to store the chat history of each user separately.

store = {} # this is a dictionary that will be used to store the chat history of each user based on the session id. The key of the dictionary will be the session id and the value will be the chat history of that session.

def get_session_history(session_id : str) -> BaseChatMessageHistory:
    ## this function is used to get the chat history of a user based on the session id. 

    if session_id not in store:
        store[session_id]=ChatMessageHistory() ## if the session id is not present in the store, then we will create a new chat history for that session id and store it in the store.
    
    return store[session_id]


with_message_history = RunnableWithMessageHistory(model,get_session_history)
# this function is used to create a runnable with message history. It takes the model and the get_message_history function as input and returns a runnable that can be used to interact with the model while keeping track of the chat history based on the session id.


In [23]:
config = {
    "configurable":{
        "session_id":"chat1"
    }
}

In [ ]:
response4 = with_message_history.invoke(
    
    HumanMessage(
        content="Hey, I am Pratham Raj and Currently I am software engineer at Mindsprint"
    ),
    config=config # its meaning is to pass the session id in the config parameter of the invoke function, so that the model can keep track of the chat history based on the session id.
    
)

In [29]:
response4.content

'Nice to meet you, Pratham! 👋 It’s great to hear you’re a software engineer at Mindsprint. How can I help you today? Whether it’s a technical question, career advice, brainstorming ideas, or anything else, just let me know!'

In [30]:
print(response4.content)

Nice to meet you, Pratham! 👋 It’s great to hear you’re a software engineer at Mindsprint. How can I help you today? Whether it’s a technical question, career advice, brainstorming ideas, or anything else, just let me know!


In [31]:
response5 = with_message_history.invoke(
    
    HumanMessage(
        content="Who am I?"
    ),
    config=config 
    
)

In [32]:
response5.content

'You’re\u202fPratham\u202fRaj, and you work as a software engineer at\u202fMindsprint.'

In [33]:
response6 = with_message_history.invoke(
    
    HumanMessage(
        content="What is my name?"
    ),
    config=config 
    
)
response6.content

'Your name is **Pratham\u202fRaj**.'

In [34]:
response7 = with_message_history.invoke(
    
    HumanMessage(
        content="What is my age?"
    ),
    config=config 
    
)
response7.content

'I don’t have any information about your age. If you’d like to share it, feel free to let me know!'

In [35]:
# change the config --> means changing the session id 
config2 = {
    "configurable":{
        "session_id":"chat2"
    }
}

In [36]:
response8 = with_message_history.invoke(
    
    HumanMessage(
        content="What is my name?"
    ),
    config=config2
    
)
response8.content

'I don’t have any information about your name. If you’d like me to address you in a particular way, just let me know!'

In [37]:
response9 = with_message_history.invoke(
    
    HumanMessage(
        content="Hey! my name is Shivam Kumar"
    ),
    config=config2
    
)
response9.content

'Nice to meet you, Shivam Kumar! How can I assist you today?'

In [38]:
response10 = with_message_history.invoke(
    
    HumanMessage(
        content="What is my name?"
    ),
    config=config2
    
)
response10.content

'Your name is Shivam\u202fKumar.'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
# this line is used to import the ChatPromptTemplate and MessagesPlaceholder classes from the langchain_core.prompts library, which allows us to create a chat prompt template that can be used to interact with the model while keeping track of the chat history based on the session id.
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system","You are a helpful assistant. Answer all the question to the nest of your ability."
        ),
        # this system message is used to set the behavior of the assistant. It tells the assistant that it is a helpful assistant and it should answer all the questions to the best of its ability.
        MessagesPlaceholder(variable_name="messages")
        # this line is used to add a placeholder for the messages in the chat prompt template. The variable name "messages" will be used to refer to the chat history in the prompt template.
    ]
)

chain = prompt|model

In [40]:
response11 = chain.invoke(
    {
        "messages":[
            HumanMessage(
                content="Hey! My name is Shivam Kumar"
            )
        ]
    }
)
response11.content

'Hey Shivam! Nice to meet you. How can I help you today?'

In [41]:
response12 = chain.invoke(
    {
        "messages":[
            HumanMessage(
                content="Hey!What is my name?"
            )
        ]
    }
)
response12.content

'I don’t actually know your name—unless you’ve shared it with me before, I don’t have that information. If you’d like, feel free to tell me your name and I’ll be happy to use it!'

In [42]:
with_message_history = RunnableWithMessageHistory(chain,get_session_history)

In [43]:
config3 = {
    "configurable":{
        "session_id":"chat3"
    }
}

In [44]:
response13 = with_message_history.invoke(
    {
        "messages": [
            HumanMessage(
                content="Hey! My name is Shivam Kumar"
            )
        ]
    },
    config=config3
)
response13.content

'Hello Shivam! Nice to meet you. How can I assist you today?'

In [ ]:
# Add more complexity like we can get output in particular language for this we can also give input for language based on choise

In [45]:

prompt2 = ChatPromptTemplate.from_messages(
    [
        (
            "system","You are a helpful assistant. Answer all the question to the nest of your ability in {language}."
        ),
        
        MessagesPlaceholder(variable_name="messages")
        
    ]
)

chain2 = prompt2|model

In [47]:
response14 = chain2.invoke(
    {
        "messages":[
            HumanMessage(
                content="Hey! My name is Shivam Kumar"
            )
        ],
        "language":"Hindi"
    }
)
response14.content

'नमस्ते शिवम कुमार! आपसे मिलकर खुशी हुई। मैं आपका मददगार सहायक हूँ। आपसे कैसे मदद कर सकता हूँ?'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [48]:
with_message_history  = RunnableWithMessageHistory(
    chain2,
    get_session_history,
    input_messages_key="messages"
)

In [49]:
config4 = {
    "configurable":{
        "session_id":"chat4"
    }
}

In [52]:
response15 = with_message_history.invoke(
    {
        "messages":[
            HumanMessage(
                content="Hi! I am Pratham Raj"
            )

        ],
        "language":"Hindi"
        
    },
    config=config4
)

In [53]:
response15.content

'नमस्ते प्रथम राज! आपसे मिलकर खुशी हुई। मैं आपकी कैसे मदद कर सकता हूँ? 😊'

In [54]:
response16 = with_message_history.invoke(
    {
        "messages":[
            HumanMessage(
                content="What is my name?"
            )

        ],
        "language":"English"
        
    },
    config=config4
)
response16.content

'Your name is Pratham\u202fRaj.'

In [55]:
response17 = with_message_history.invoke(
    {
        "messages":[
            HumanMessage(
                content="What is my name?"
            )

        ],
        "language":"Hindi"
        
    },
    config=config4
)
response17.content

'आपका नाम प्रथम राज है। क्या मैं आपकी किसी और चीज़ में मदद कर सकता हूँ?'